In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv("../keys.env")
assert os.environ["GROQ_API_KEY"].startswith("gsk"), \
  "Please specify the GROQ_API_KEY access token in keys.env file"
 

In [3]:
import gutenberg_text_loader as gtl
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [4]:
gs = gtl.GutenbergSource()
doc = gs.load_from_url("https://www.gutenberg.org/cache/epub/46976/pg46976.txt")

2026-04-03 16:57:43,886 - INFO - Created cache directory at .cache
2026-04-03 16:57:43,887 - INFO - Fetching text from URL: https://www.gutenberg.org/cache/epub/46976/pg46976.txt
2026-04-03 16:57:45,924 - INFO - Cached content for https://www.gutenberg.org/cache/epub/46976/pg46976.txt at .cache/pg46976_1b16d8525c.txt
2026-04-03 16:57:45,943 - INFO - Cleaned Gutenberg text: removed 1204 chars from start, 18808 chars from end
2026-04-03 16:57:45,945 - INFO - Successfully loaded text from https://www.gutenberg.org/cache/epub/46976/pg46976.txt.


In [5]:
doc.text[21000:22000]

'iarch of Constantinople in\r\nthe ninth century, and from a few incidental references in his own\r\nwritings. We learn from Suidas that Dion Cassius wrote a biography of\r\nArrian; but this work is not extant. Flavius Arrianus was born near\r\nthe end of the first century of the Christian era, at Nicomedia, the\r\ncapital of Bithynia. He became a pupil of the famous Stoic philosopher\r\nEpictetus, and afterwards went to Athens, where he received the surname\r\nof the “younger Xenophon,” from the fact that he occupied the same\r\nrelation to Epictetus as Xenophon did to Socrates.[1] Not only was he\r\ncalled Xenophon by others, but he calls himself so in _Cynegeticus_ (v.\r\n6); and in _Periplus_ (xii. 5; xxv. 1), he distinguishes Xenophon by\r\nthe addition _the elder_. Lucian (_Alexander_, 56) calls Arrian simply\r\n_Xenophon_. During the stay of the emperor Hadrian at Athens, A.D. 126,\r\nArrian gained his friendship. He accompanied his patron to Rome, where\r\nhe received the Roman

In [6]:
print(doc.id_)

9cf39888-17dd-404c-91be-aaa762a69e48


In [10]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import Document

class Indexer:
    """
    A class to load documents into LlamaIndex using BM25
    Attributes:
      chunk_size (int): Size of text chunks for processing.
      chunk_overlap (int): Overlap between text chunks.
      docstore (SimpleDocumentStore): Document store for storing processed documents.
    """

    def __init__(self,
                 cache_dir: str = './.cache',
                 chunk_size = 1024,
                 chunk_overlap = 20):
        """
        Initialize the Indexer

        Args:
          chunk_size(int): Size of text chunks for processing. Defaults to 1024.
          chunk_overlap (int): Overlap between text chunks. Defaults to 20.
        """

        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

        self.docstore = SimpleDocumentStore()
        self.node_parser = SentenceSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap
        )

        logger.info("Indexer initialized")

    def add_document_to_index(self, document: Document):
        nodes = self.node_parser.get_nodes_from_documents([document])
        self.docstore.add_documents(nodes)
        logger.info(f"Successfully loaded text from {document.id_} -- {len(nodes)} created")
    
    def get_docstore(self) -> SimpleDocumentStore:
        return self.docstore

In [11]:
index = Indexer(chunk_size=100, chunk_overlap=20)
index.add_document_to_index(doc)

2026-04-03 17:13:13,207 - INFO - Indexer initialized
2026-04-03 17:13:16,156 - INFO - Successfully loaded text from 9cf39888-17dd-404c-91be-aaa762a69e48 -- 6120 created


In [12]:
from llama_index.retrievers.bm25 import BM25Retriever
retriever = BM25Retriever.from_defaults(
    docstore=index.get_docstore(),
    similarity_top_k=5
)

2026-04-03 17:58:32,370 - DEBUG - Building index from IDs objects


In [14]:
from llama_index.core.response.notebook_utils import display_source_node
retrieved_nodes = retriever.retrieve("Describe the relationship between Alexander and Diogenes")
for node in retrieved_nodes:
    display_source_node(node, 1024)

2026-04-03 18:09:02,593 - INFO - Failed to extract font properties from /home/sharofiddin/.local/share/fonts/Sauce Code Pro Nerd Font Complete.ttf: Can not load face (invalid stream operation; error code 0x55)
2026-04-03 18:09:02,632 - INFO - Failed to extract font properties from /usr/share/fonts/truetype/noto/NotoColorEmoji.ttf: Can not load face (unknown file format; error code 0x2)
2026-04-03 18:09:02,926 - WARNING - Matplotlib is building the font cache; this may take a moment.
2026-04-03 18:09:05,426 - INFO - generated new fontManager


**Node ID:** 9aa5e23e-0c21-4c04-92ba-4d0992635782<br>**Similarity:** 4.305605888366699<br>**Text:** But Diogenes said that he
wanted nothing else, except that he and his attendants would stand out
of the sunlight. Alexander is said to have expressed his admiration
of Diogenes’s conduct.<br>

**Node ID:** 4a359e83-1b0d-415b-b0be-dd872375aba3<br>**Similarity:** 4.107877731323242<br>**Text:** 100 stades; and most of it is the mean between
these breadths.[642] This river Indus Alexander crossed at daybreak
with his army into the country of the Indians; concerning whom, in
this history I have described neither what laws they enjoy,<br>

**Node ID:** aad8aee4-26a7-4b97-90a2-e52a54df7a70<br>**Similarity:** 3.6830084323883057<br>**Text:** 32). Alexander said: “If I were
not Alexander, I should like to be Diogenes.” Cf. _Arrian_, i. 1;
Plutarch (_de Fortit. Alex._, p. 331).<br>

**Node ID:** 76a2892e-4a7d-4338-9e5a-cb86e5a6c5bb<br>**Similarity:** 3.4497487545013428<br>**Text:** Alexander is said to have expressed his admiration
of Diogenes’s conduct.[832] Thus it is evident that Alexander was
not entirely destitute of better feelings; but he was the slave of
his insatiable ambition.<br>

**Node ID:** 99383012-0b3c-495c-b1e5-88bcd5ea915a<br>**Similarity:** 3.242121934890747<br>**Text:** He also ascertained that for
the present Bessus held the supreme command, both on account of his
relationship to Darius and because the war was being carried on in his
viceregal province. Hearing this,<br>

In [23]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.llms.groq import Groq

llm = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
    model='moonshotai/kimi-k2-instruct-0905',
    temperature=0.2)


In [24]:
from llama_index.core.llms import ChatMessage
messages = [
    ChatMessage(
        role="system",
        content="Use the following text to answer the given question."
    )
]

messages += [
    ChatMessage(role="system", content=node.text) for node in retrieved_nodes
]

messages += [
    ChatMessage(role="user", content="Describe the relationship between Alexander and Diogenes.")
]

response = llm.chat(messages)
print(response)

2026-04-03 18:29:20,987 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


assistant: Alexander and Diogenes had a relationship of mutual, if asymmetrical, respect. When Alexander visited Corinth, he sought out the famously ascetic philosopher, who showed no interest in the conqueror’s power or wealth. Diogenes merely asked Alexander and his attendants to step out of his sunlight. Instead of taking offense, Alexander admired Diogenes’s independence and is reported to have said, “If I were not Alexander, I should like to be Diogenes,” acknowledging the philosopher’s freedom from ambition that he himself could not escape.


In [25]:
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever, llm=llm
)

response = query_engine.query("Describe the relationship between Alexander and Diogenes.")
response = {
    "answer": str(response),
    "source_nodes": response.source_nodes
}
print(response['answer'])

2026-04-03 18:33:27,719 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander admired Diogenes’ way of life, openly praising his conduct and remarking that, were he not Alexander, he would choose to be Diogenes.


In [ ]:
for node in response['source_nodes']:
    print(node)